# Cirrus and halo modeling: Cutout ddV

Inject a smooth galaxy into the observed Cutout field, model the cirrus, and
compare the recovered light with the known input. We use a 36″ FIR likelihood
and one existing representative seed pair. This is a new run at 36″; the seed
was selected from the earlier 18″ ensemble and is not reselected here.

## 1. Set up the inputs and runtime

Open this notebook from a CirrusMachine checkout. In Colab, select a GPU runtime
and copy or clone the checkout, including the optional `product/examples/cutout_ddV`
input bundle. Mount Drive if needed. The setup discovers an editable installation
or a checkout above the current directory; environment overrides are optional.
The `[examples]` extra installs Jupyter and widget dependencies. Scientific input
files are separate from pip dependencies and remain outside the wheel.
The Cheng `scattering` backend is bundled; PyTorch uses CUDA when available and
otherwise CPU. A full CPU run can be slow.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys

# Colab with Drive storage: uncomment these lines if needed.
# from google.colab import drive
# drive.mount("/content/drive")

candidates = []
if os.environ.get("CIRRUSMACHINE_REPO"):
    candidates.append(Path(os.environ["CIRRUSMACHINE_REPO"]).expanduser())
spec = importlib.util.find_spec("cirrusmachine")
if spec is not None and spec.origin:
    candidates.extend(Path(spec.origin).resolve().parents)
candidates.extend([Path.cwd(), Path.cwd() / "CirrusMachine", *Path.cwd().parents])
REPO_DIR = next((p for p in candidates
                 if (p / "src/cirrusmachine").is_dir()
                 and (p / "notebooks/cutout_ddV_helpers.py").is_file()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Open from the checkout or set CIRRUSMACHINE_REPO.")
DATA_DIR = Path(os.environ.get(
    "CIRRUSMACHINE_DATA", str(REPO_DIR / "product/examples/cutout_ddV")
)).expanduser()
OUTPUT_ROOT = REPO_DIR / "results/cutout_ddV"

In [2]:
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[examples]"
])

0

In [11]:
from dataclasses import replace
from datetime import datetime, timezone
from uuid import uuid4
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
from astropy.io import fits
from cirrusmachine import modeling as cm

sys.path.insert(0, str(REPO_DIR / "notebooks"))
from cutout_ddV_helpers import (
    load_inputs, show_image, plot_surface_brightness, plot_cumulative_flux,
)

logger = cm.configure_logging("INFO")  # Optional: DEBUG for extra diagnostics.
config = cm.load_config(REPO_DIR / "configs/cutout_ddV.yaml")
if config.device == "auto":
    config = replace(config, device="cuda" if torch.cuda.is_available() else "cpu")
if config.device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

# Optional: add cm.DrawSeeds(broad=..., texture=...) entries to config.draws.
# The default pair is the existing representative realization.
logger.info("Device: %s%s", config.device, f" ({torch.cuda.get_device_name(0)})" if config.device == "cuda" else "")
logger.info("Run: %d realization(s), %d steps", len(config.draws), config.steps)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID

[INFO] Device: cpu
[INFO] Run: 1 realization(s), 150 steps


The optional input bundle contains `optical.fits`, `masks.fits`, `fir.fits`,
`psfs.npz`, and `inputs.json` (about 30 MB). These preserve observed data,
source/target masks, native FIR samples and uncertainties, and detector/working
PSFs. Checksums verify the files; provenance records filenames without local paths.
Model caches are optional.

In [4]:
inputs = load_inputs(DATA_DIR)
meta = inputs["metadata"]
if not np.isclose(config.beam_FWHM, meta["effective_FWHM"]):
    raise ValueError("The configured FIR resolution differs from the input manifest.")
logger.info("Loaded optical %s and FIR %s; input checksums verified",
            inputs["optical"].shape, inputs["FIR_image"].shape)

[INFO] Loaded optical (2000, 2000) and FIR (456, 440); input checksums verified


## 2. Prepare the field and inject a galaxy

The two de Vaucouleurs components share a center, axis ratio 0.6 and position
angle 30°, with effective radii 8″ and 32″. `Ie` is intensity at `Re`; alternatively
supply central intensity `I0 = Ie exp(b_n)`. Convolve once with the detector PSF
and mean-bin to 0.9″ pixels. The input optical WCS is 0.3″/pixel; the notebook
applies 3×3 mean binning to reach the 0.9″ working WCS. Injection is sampled
at 0.1″ and mean-binned 9×9 to that working grid, not 10×10. Source masks
stay fixed; truth stays separate.

The [ERO VIS calibration](https://doi.org/10.1051/0004-6361/202450803) is
`VIS_ZP = 30.13` for flux in ADU at 0.1″ sampling. Under the retained mean-intensity
convention, `SB_ZP` converts ADU to mag arcsec⁻².

The adopted convention gives approximately
24 mag arcsec⁻² at 20″ for the combined intrinsic galaxy.

In [ ]:
components = [
    {"n": 4, "Re_arcsec": 8.0, "Ie": 11.73},
    {"n": 4, "Re_arcsec": 32.0, "Ie": 0.51},
]
center_native = (235.0 / 0.3, 175.0 / 0.3)
axis_ratio, PA_deg = 0.6, 30.0
VIS_ZP = 30.13
intensity_pixel_scale = 0.1  # Original VIS units retained by assumed mean binning.
SB_ZP = cm.surface_brightness_zeropoint(VIS_ZP, intensity_pixel_scale)
logger.info("VIS ZP = %.2f; intensity scale = %.1f arcsec; SB ZP = %.2f",
            VIS_ZP, intensity_pixel_scale, SB_ZP)

background = cm.prepare_field_data(
    inputs["optical"], inputs["source_mask"], inputs["target_mask"],
    inputs["PSF_working"], center_xy=center_native,
    axis_ratio=axis_ratio, PA_deg=PA_deg,
    native_pixel_scale=meta["native_pixel_scale"], bin_factor=3,
    canvas_size=1024, exclusion_radius=150, WCS=inputs["optical_WCS"],
)
if not np.isclose(background.pixel_scale, config.pixel_scale):
    raise ValueError("Working pixel scale differs from the model configuration.")
if not np.isclose(background.pixel_scale / 9, meta["detector_psf_pixel_scale"]):
    raise ValueError("Detector PSF sampling differs from the injection grid.")
injection = cm.prepare_injection(
    background, components, axis_ratio=axis_ratio, PA_deg=PA_deg,
    PSF_native=inputs["PSF_detector"], oversample=9, device=config.device,
)
field = injection.field
crop = field.crop
valid = field.source_clean[crop]
radius = field.radius[crop]
logger.info("Working crop %s at %.1f arcsec/pixel",
            field.image[crop].shape, field.pixel_scale)

[INFO] VIS ZP 30.13; intensity scale 0.1 arcsec; SB ZP 25.13
[INFO] Working crop (666, 666) at 0.9 arcsec/pixel; 204293 retained cirrus pixels


## 3. Measure the cirrus statistics

Estimate noise and texture from the retained optical pixels, accounting for
mask support. Reference fields calibrate these measurements; they are not extra
optimized outputs. Reuse a matching cache on later runs; delete it or change its
path after changing the inputs or statistical settings.

In [12]:
CACHE_FILE = OUTPUT_ROOT / "cache/statistics.npz"
if CACHE_FILE.is_file():
    statistics = cm.load_calibration(CACHE_FILE, field, config)
    logger.info("Loaded statistical calibration matching the data, masks and settings.")
else:
    statistics = cm.calibrate_statistics(field, config)
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    cm.save_calibration(statistics, CACHE_FILE)

[INFO] Loaded statistical calibration matching the data, masks and settings.


## 4. Condition on the FIR map and initialize the cirrus

Use the adopted 36″ effective FIR resolution to rebuild the likelihood and
its calibration covariance from native, unfilled samples. The FIR guide sets
large-scale structure; a conditional draw supplies the optical-scale detail.

In [13]:
beam = cm.prepare_FIR_constraint(
    field, inputs["FIR_image"], inputs["FIR_error"],
    inputs["FIR_coverage"], inputs["FIR_mask"], inputs["FIR_WCS"], config,
    effective_FWHM=meta["effective_FWHM"],
)
prior = cm.build_cirrus_prior(field, beam, statistics, config)
initial = cm.initialize_cirrus(prior, config)
logger.info("FIR likelihood: %d native samples, %d calibration samples; %d initial draw(s)",
            len(beam["target"]), int(beam["cal"].sum()), len(initial))

[INFO] FIR likelihood: 875 native samples, 252 calibration samples; 1 initial draw(s)


## 5. Model the cirrus and recover the halo

Alternate cirrus optimization with smooth halo and sky updates. Only the
injected observations and masks enter the model; the known galaxy is used
later for comparison. The default run produces one realization.

In [ ]:
results = cm.run_cirrus_halo_modeling(
    field, prior, initial, config,
)
result = results[0]  # Existing representative seed pair; no ensemble selection.
logger.debug("Completed seed pair: %s", result.seeds)

## 6. Compare surface brightness and cumulative flux

Measure truth, direct and corrected images on the same source-clean mask and
elliptical apertures. Truth has zero added sky; direct light uses an annulus
estimate, and corrected light uses the model's sky from that same 180–220″
annulus. Galaxy wings in the sky annulus can bias recovery. Cumulative flux is
a signed pixel sum without filling source holes; no uncertainty is inferred
from this single realization.

In [ ]:
images = {
    "Truth": injection.truth[crop],
    "Direct": field.image[crop],
    "Corrected": result.corrected,
}
direct_sky = cm.estimate_sky(
    images["Direct"], valid, radius, config.sky_annulus, device=config.device,
)
skies = {"Truth": 0.0, "Direct": direct_sky["value_ADU"], "Corrected": result.sky}
edges = np.concatenate(([0.0], np.geomspace(1.8, 150.0, 35)))
profiles, cumulative = {}, {}
for name, image in images.items():
    profiles[name] = cm.measure_profile(
        image, valid, radius, edges, sky=skies[name],
        SB_ZP=SB_ZP, device=config.device,
    )
    cumulative[name] = cm.measure_cumulative_light(
        image, valid, radius, edges[1:], sky=skies[name],
        pixel_scale=field.pixel_scale, device=config.device,
    )
logger.info("Sky (ADU): truth %.4g, direct %.4g, corrected %.4g",
            skies["Truth"], skies["Direct"], skies["Corrected"])

## 7. Inspect the images and profiles

These plots use the saved-in-memory measurements and maps; rerunning them does
not repeat modeling. Excluded data pixels are gray; truth and model maps are shown without holes. Image limits are adjustable. Surface-brightness bins with
nonpositive intensity are omitted from the magnitude plot.

In [ ]:
mu, std = field.normalization
initial_image = mu + std * initial[0].image.detach().cpu().numpy()[crop]
fig_inputs, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
show_image(background.image[crop], valid=valid, title="Observed field", ax=axes[0])
show_image(injection.truth[crop], title="Injected galaxy", ax=axes[1])
show_image(images["Direct"], valid=valid, title="Field + galaxy", ax=axes[2])
plt.show()

fig_prior, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
show_image(mu + std * beam["backbone"][crop], title="FIR guide", ax=axes[0])
show_image(initial_image, title="Initial cirrus", ax=axes[1])
plt.show()

In [ ]:
fig_models, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
show_image(result.cirrus, title="Modeled cirrus", ax=axes[0])
show_image(result.corrected - result.sky, valid=valid, title="Corrected, sky subtracted", ax=axes[1])
difference = result.corrected - result.sky - images["Truth"]
limit = np.nanpercentile(np.abs(difference[valid]), 99.5)
show_image(difference, valid=valid, title="Corrected − truth", ax=axes[2],
           limits=(-limit, limit), cmap="RdBu_r")
plt.show()

In [ ]:
fig_photometry, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
plot_surface_brightness(profiles, ax=axes[0])
plot_cumulative_flux(cumulative, ax=axes[1])
plt.show()

## 8. Save this run

Save model maps, numerical measurements, injection inputs, configuration and
figures to a new directory. Original data and earlier runs are kept intact.

In [ ]:
cm.save_modeling_results(results, OUTPUT_DIR, config, field=field)
cm.save_calibration(statistics, OUTPUT_DIR / "statistics.npz")
np.savez_compressed(OUTPUT_DIR / "fir_constraint.npz", **beam)
np.savez_compressed(
    OUTPUT_DIR / "photometry.npz",
    **{f"{name.lower()}_profile_{key}": value
       for name, measured in profiles.items() for key, value in measured.items()},
    **{f"{name.lower()}_cumulative_{key}": value
       for name, measured in cumulative.items() for key, value in measured.items()},
)
header = field.WCS.slice(crop).to_header()
header["BUNIT"] = "ADU"
for name, image in {"truth": images["Truth"], "direct": images["Direct"],
                    "initial": initial_image}.items():
    fits.writeto(OUTPUT_DIR / f"{name}.fits", image.astype("f4"), header)
fits.writeto(OUTPUT_DIR / "source_clean.fits", valid.astype("u1"), field.WCS.slice(crop).to_header())
metadata = {
    "inputs": inputs["metadata"], "injection": injection.parameters,
    "center_native_xy": list(center_native), "SB_ZP": SB_ZP,
    "VIS_ZP": VIS_ZP, "intensity_pixel_scale": intensity_pixel_scale,
    "parent_binning": "assumed mean; original parent header lacks binning metadata",
    "skies_ADU": skies, "device_used": config.device,
    "photometry_geometry": "elliptical semimajor apertures on identical source-clean support",
    "displayed_draw": 0,
    "selection": "first configured draw; default is the existing draw3 seed pair",
}
(OUTPUT_DIR / "example.json").write_text(json.dumps(metadata, indent=2))
for name, figure in {"inputs": fig_inputs, "prior": fig_prior,
                     "model": fig_models, "photometry": fig_photometry}.items():
    figure.savefig(OUTPUT_DIR / f"{name}.png", dpi=150)
logger.info("Saved maps, photometry, configuration and figures to results/cutout_ddV/%s", RUN_ID)